# Notebook 01 — Residue Manifold Construction

**Repo:** `residue-manifold-learning`  
**Purpose:** construct the baseline mod30 residue manifold, identify the eight prime residue lanes, and export reusable data/figures for later notebooks.

This notebook is intentionally simple: no ML, no CGCS metric, no sparse autoencoders yet. It defines the object that later notebooks learn from.


## Outputs

This notebook writes:

- `data/residues_mod30.csv`
- `data/primes_mod30.csv`
- `data/residue_lane_summary_mod30.csv`
- `figures/residue_histogram_mod30.png`
- `figures/residue_circle_mod30.png`
- `figures/residue_lane_matrix_mod30.png`


In [ ]:
# Standard imports
from pathlib import Path
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from sympy import primerange
except ImportError as exc:
    raise ImportError("This notebook needs sympy. Install with: pip install sympy") from exc

# Make imports work whether running from repo root or from notebooks/
import sys
repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
src_path = repo_root / "src"
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

DATA_DIR = repo_root / "data"
FIG_DIR = repo_root / "figures"
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

MOD = 30
N_MAX = 10_000


## 1. Construct residue space

The baseline space is the finite cyclic residue system `Z / 30Z`. For each integer `n`, the residue lane is `n mod 30`.


In [ ]:
n_values = np.arange(N_MAX + 1)
residues = n_values % MOD

df_residues = pd.DataFrame({
    "n": n_values,
    "mod": MOD,
    "residue": residues,
})

df_residues.head()

## 2. Generate primes and prime residues

For primes above 5, all prime residues mod30 must be coprime to 30. This leaves eight possible lanes.


In [ ]:
primes = np.array(list(primerange(2, N_MAX + 1)), dtype=int)
prime_residues = primes % MOD

df_primes = pd.DataFrame({
    "p": primes,
    "mod": MOD,
    "residue": prime_residues,
    "is_small_prime_divisor_of_30": np.isin(primes, [2, 3, 5]),
})

# Lanes for primes larger than the small prime divisors of 30.
df_prime_lanes = df_primes.loc[~df_primes["is_small_prime_divisor_of_30"]].copy()
valid_lanes = sorted(df_prime_lanes["residue"].unique().tolist())

valid_lanes

In [ ]:
expected_lanes = [r for r in range(MOD) if math.gcd(r, MOD) == 1]
print("Expected coprime lanes:", expected_lanes)
print("Observed prime lanes:  ", valid_lanes)
assert valid_lanes == expected_lanes
print(f"Confirmed: {len(valid_lanes)} valid lanes out of {MOD} residues.")

## 3. Lane summary table

This table is useful for later notebooks. It separates all residue lanes into valid prime lanes and excluded lanes.


In [ ]:
lane_counts = df_prime_lanes["residue"].value_counts().reindex(range(MOD), fill_value=0).sort_index()

df_lane_summary = pd.DataFrame({
    "mod": MOD,
    "residue": np.arange(MOD),
    "gcd_residue_mod": [math.gcd(r, MOD) for r in range(MOD)],
    "is_coprime_lane": [math.gcd(r, MOD) == 1 for r in range(MOD)],
    "prime_count_excluding_2_3_5": lane_counts.values,
})

df_lane_summary["lane_label"] = np.where(
    df_lane_summary["is_coprime_lane"],
    "valid_prime_lane",
    "excluded_lane",
)

df_lane_summary

## 4. Density baseline

The mod30 prime lane density is the fraction of residue classes that remain possible after excluding residues sharing a factor with 30.


In [ ]:
lane_density = len(valid_lanes) / MOD
print(f"Valid lanes: {len(valid_lanes)} / {MOD}")
print(f"Lane density: {lane_density:.6f}")

## 5. Figure — prime residue histogram

This figure shows the eight valid lanes and the excluded lanes. It should be the simplest baseline plot for the paper introduction.


In [ ]:
counts_all = df_prime_lanes["residue"].value_counts().reindex(range(MOD), fill_value=0).sort_index()

plt.figure(figsize=(10, 4.8))
plt.bar(counts_all.index.astype(str), counts_all.values)
plt.title("Prime Residue Lanes mod 30")
plt.xlabel("Residue class r mod 30")
plt.ylabel("Prime count up to N, excluding 2, 3, 5")
plt.xticks(rotation=0)
plt.tight_layout()

hist_path = FIG_DIR / "residue_histogram_mod30.png"
plt.savefig(hist_path, dpi=180)
plt.show()

hist_path

## 6. Figure — circular residue manifold

Embedding each residue class as an angle on the unit circle makes the lane structure visible as a discrete angular manifold.


In [ ]:
theta = np.linspace(0, 2 * np.pi, 600)
all_angles = 2 * np.pi * np.arange(MOD) / MOD
valid_angles = 2 * np.pi * np.array(valid_lanes) / MOD

plt.figure(figsize=(7, 7))
plt.plot(np.cos(theta), np.sin(theta), linewidth=1, alpha=0.4)

# all residues as small markers
plt.scatter(np.cos(all_angles), np.sin(all_angles), s=35, alpha=0.35, label="all residues")

# valid lanes as larger markers
plt.scatter(np.cos(valid_angles), np.sin(valid_angles), s=120, label="valid prime lanes")

for r, a in zip(range(MOD), all_angles):
    radius = 1.13 if r in valid_lanes else 1.07
    plt.text(radius * np.cos(a), radius * np.sin(a), str(r), ha="center", va="center", fontsize=8)

plt.title("Residue Manifold: valid prime lanes in Z/30Z")
plt.axis("equal")
plt.axis("off")
plt.legend(loc="upper right")
plt.tight_layout()

circle_path = FIG_DIR / "residue_circle_mod30.png"
plt.savefig(circle_path, dpi=180)
plt.show()

circle_path

## 7. Figure — lane indicator matrix

This compact matrix is useful later for comparing learned basis features against ground-truth residue lanes.


In [ ]:
lane_indicator = np.zeros((1, MOD), dtype=int)
lane_indicator[0, valid_lanes] = 1

plt.figure(figsize=(11, 1.8))
plt.imshow(lane_indicator, aspect="auto")
plt.yticks([0], ["valid lane"])
plt.xticks(range(MOD), range(MOD), fontsize=8)
plt.title("Ground-truth mod30 valid prime lane indicator")
plt.xlabel("Residue class r mod 30")
plt.tight_layout()

matrix_path = FIG_DIR / "residue_lane_matrix_mod30.png"
plt.savefig(matrix_path, dpi=180)
plt.show()

matrix_path

## 8. Save reusable data

Later notebooks should load these CSVs instead of reconstructing the baseline every time.


In [ ]:
residues_path = DATA_DIR / "residues_mod30.csv"
primes_path = DATA_DIR / "primes_mod30.csv"
lane_summary_path = DATA_DIR / "residue_lane_summary_mod30.csv"

df_residues.to_csv(residues_path, index=False)
df_primes.to_csv(primes_path, index=False)
df_lane_summary.to_csv(lane_summary_path, index=False)

print(residues_path)
print(primes_path)
print(lane_summary_path)

## 9. Notebook 01 claim

Prime numbers under mod30 occupy a fixed set of eight coprime residue lanes:

`1, 7, 11, 13, 17, 19, 23, 29`

This creates the baseline residue manifold used by later notebooks. The next step is **Notebook 02 — Constraint Sampling**, where uniform sampling is compared against structure-aware sampling.
